**PASS: fresh worked-example execution.** All saved outputs below came from this notebook's actual code. Run UTC: `2026-11-05T07:19:59.822650+00:00`; executed 8/8 code cells; 8 output objects; 0 saved errors. No execution errors are allowed or suppressed. These are provided worked examples; unanswered learner practice is separate Markdown. The synthetic scope still applies.
<!-- canonical-execution-status:end -->

# Tier 2 — completed worked exercise examples

**Complete reference solutions.** The historical filename `Tier2_Student.ipynb` is retained for existing links. This notebook and its local implementation are now fully worked examples, with no executable blank tasks. They are suitable for self-study and inspection; they are not an independent student submission.

For a dedicated presentation route, open `notebooks/Tier2_Instructor.ipynb` from the package. Independent prompts are in `teaching/STUDENT_PRACTICE.md`.

## Goal, motivation and contracts

A correct attention equation is only the beginning: code must preserve axes, visibility and valid supervision under padding. This completed reference lab challenges the implementation with an independent API, future perturbations, rectangular sources, cached offsets and empty-supervision edge cases.

Prerequisites: Tier 1 reasoning, tensor slicing and the idea that gradients update learned parameters. Budget for guided study: 2–3 hours. All functions are implemented; every runnable cell is a worked demonstration. The tests check this reference implementation rather than independent student completion.

In [1]:
from pathlib import Path
import importlib.util
import inspect
import math
import torch
import torch.nn.functional as F

# Find this package from either notebooks/, student_exercises/, or package root.
package_candidates = [Path.cwd(), *Path.cwd().parents]
PACKAGE_ROOT = next(
    candidate for candidate in package_candidates
    if (candidate / "src/vgu_transformers.py").is_file()
    and (candidate / "notebooks/student_exercises/student_impl.py").is_file()
)
implementation_path = PACKAGE_ROOT / "notebooks/student_exercises/student_impl.py"
module_spec = importlib.util.spec_from_file_location("vgu_completed_tier12", implementation_path)
work = importlib.util.module_from_spec(module_spec)
module_spec.loader.exec_module(work)
assert Path(work.__file__).resolve() == implementation_path.resolve()
torch.manual_seed(9)
torch.set_num_threads(1)
torch.set_printoptions(precision=4, sci_mode=False)
print("Completed implementation:", implementation_path.relative_to(PACKAGE_ROOT).as_posix())
print("PyTorch:", torch.__version__)


Completed implementation: notebooks/student_exercises/student_impl.py
PyTorch: 2.10.0+cpu


## Read the implementation before testing it

The four short functions are displayed from the **actual imported local module**. `attention` computes scaled Q/K scores, applies an additive mask, normalizes over keys and mixes V. It handles a fully blocked row by constructing a finite softmax input for that row and selecting zero weights under the declared contract. It does not catch errors or turn a failed check into a success.

`causal_mask` defines visible key indices, `next_token_pairs` shifts supervision, and `masked_cross_entropy` excludes the explicitly chosen padded targets. These operations serve different purposes.

In [2]:
for function in [work.attention, work.causal_mask, work.next_token_pairs, work.masked_cross_entropy]:
    print(inspect.getsource(function))


def attention(q, k, v, mask=None):
    """Return (mixed values, weights) for q [...,Tq,d], k [...,Tk,d], v [...,Tk,dv].

    Each valid query distributes weight across keys. Our declared policy for a
    fully blocked query is zero weights/output, with finite zero gradients for
    that query. We construct a safe softmax input before normalization; we do
    not discard exceptions or conceal invalid results after computation.
    """
    scores = (q @ k.transpose(-2, -1)) / math.sqrt(q.shape[-1])
    if mask is not None:
        if mask.dtype == torch.bool:
            raise TypeError("This API uses a floating additive 0/-inf mask.")
        scores = scores + mask
    has_visible_key = ~torch.isneginf(scores).all(dim=-1, keepdim=True)
    safe_scores = torch.where(has_visible_key, scores, torch.zeros_like(scores))
    weights = torch.softmax(safe_scores, dim=-1)
    weights = torch.where(has_visible_key, weights, torch.zeros_like(weights))
    return weights @ v, weights

def causal_ma

## 2A — Compare with an independent attention API

A square tensor can hide a transpose mistake. Here Q has `[2,3,4,8]`, K `[2,3,7,8]` and V `[2,3,7,5]`. Scores/weights must be `[2,3,4,7]` and output `[2,3,4,5]`.

The expected behavior is numerical agreement with PyTorch's scaled-dot-product attention and row sums of one over the seven keys. Small floating-point differences are compared with a stated tolerance; they are printed rather than described as exact symbolic equality.

In [3]:
g = torch.Generator().manual_seed(302)
q = torch.randn(2, 3, 4, 8, generator=g)
k = torch.randn(2, 3, 7, 8, generator=g)
v = torch.randn(2, 3, 7, 5, generator=g)
actual, weights = work.attention(q, k, v)
reference = F.scaled_dot_product_attention(q, k, v, dropout_p=0.0)
torch.testing.assert_close(actual, reference, atol=2e-6, rtol=2e-6)
torch.testing.assert_close(weights.sum(-1), torch.ones(2, 3, 4))
assert weights.shape == (2, 3, 4, 7) and actual.shape == (2, 3, 4, 5)
print("Weight / output shapes:", tuple(weights.shape), tuple(actual.shape))
print("Maximum absolute reference difference:", float((actual-reference).abs().max()))
print("Valid row-sum range:", float(weights.sum(-1).min()), float(weights.sum(-1).max()))


Weight / output shapes: (2, 3, 4, 7) (2, 3, 4, 5)
Maximum absolute reference difference: 2.384185791015625e-07
Valid row-sum range: 0.9999999403953552 1.0000001192092896


## 2B — Test causality through behavior

If only future keys/values change, earlier causal outputs must stay unchanged. That is stronger evidence than merely drawing a triangular mask. In this five-position example, alter positions 3 and 4, then compare output positions 0–2.

For contrast, the same computation **without** a causal mask can change those earlier outputs. We compute that contrast normally; there is no intentionally failing cell. It illustrates why a teacher-forced input containing future labels must not be fully visible.

In [4]:
g = torch.Generator().manual_seed(303)
q = torch.randn(1, 2, 5, 4, generator=g)
k = torch.randn(1, 2, 5, 4, generator=g)
v = torch.randn(1, 2, 5, 6, generator=g)
changed_k, changed_v = k.clone(), v.clone()
changed_k[:, :, 3:] = 100.0
changed_v[:, :, 3:] = -100.0
mask = work.causal_mask(5)
original, causal_weights = work.attention(q, k, v, mask)
changed, _ = work.attention(q, changed_k, changed_v, mask)
torch.testing.assert_close(original[:, :, :3], changed[:, :, :3])
future_positions = torch.triu(torch.ones(5, 5, dtype=torch.bool), diagonal=1)
assert causal_weights[..., future_positions].abs().sum() == 0
unmasked, _ = work.attention(q, k, v)
unmasked_changed, _ = work.attention(q, changed_k, changed_v)
unmasked_difference = float((unmasked[:, :, :3] - unmasked_changed[:, :, :3]).abs().max())
assert unmasked_difference > 1e-4
print("Earlier causal output difference:", float((original[:, :, :3]-changed[:, :, :3]).abs().max()))
print("Future attention weights:", float(causal_weights[..., future_positions].abs().max()))
print("Unmasked contrast, earlier output difference:", unmasked_difference)


Earlier causal output difference: 0.0
Future attention weights: 0.0
Unmasked contrast, earlier output difference: 101.16981506347656


## 2B extension — Cached offsets and padded keys

With three past tokens, local query zero has absolute position three. For `Tq=2,Tk=5,past=3`, the first query may read keys 0–3 and the next may read keys 0–4.

A padding-key mask serves a different purpose from causality: it prevents reading unused source positions. Perturbing only blocked padding keys/values must leave outputs unchanged. These are component checks; they do not claim that every full model automatically supports arbitrary variable-length batches.

In [5]:
offset_mask = work.causal_mask(2, t_key=5, past=3)
assert torch.isneginf(offset_mask[0, 4]) and offset_mask[1].eq(0).all()
print("Cached-offset mask:\n", offset_mask)
g = torch.Generator().manual_seed(304)
q = torch.randn(1, 2, 3, 4, generator=g)
k = torch.randn(1, 2, 5, 4, generator=g)
v = torch.randn(1, 2, 5, 6, generator=g)
padding_keys = torch.tensor([0.0, 0.0, 0.0, -math.inf, -math.inf]).view(1, 1, 1, 5)
changed_k, changed_v = k.clone(), v.clone()
changed_k[:, :, 3:] = 100.0
changed_v[:, :, 3:] = -100.0
original, _ = work.attention(q, k, v, padding_keys)
changed, _ = work.attention(q, changed_k, changed_v, padding_keys)
torch.testing.assert_close(original, changed)
print("Perturbing padded keys/values changes output by:", float((original-changed).abs().max()))


Cached-offset mask:
 tensor([[0., 0., 0., 0., -inf],
        [0., 0., 0., 0., 0.]])
Perturbing padded keys/values changes output by: 0.0


## Fully blocked queries: an explicit finite policy

Ordinary softmax of only negative infinities is undefined. This API declares that a query with no permitted keys produces zero weights and zero output. Valid rows still sum to one; a fully blocked row sums to zero.

The next cell includes one valid and one fully blocked row, then checks backward gradients. Zeroing an expected blocked row is part of the defined computation, not a way to hide unexpected execution failures.

In [6]:
g = torch.Generator().manual_seed(305)
q = torch.randn(1, 1, 2, 4, generator=g, requires_grad=True)
k = torch.randn(1, 1, 3, 4, generator=g)
v = torch.randn(1, 1, 3, 5, generator=g)
mask = torch.tensor([[0.0, 0.0, 0.0], [-math.inf, -math.inf, -math.inf]])
output, weights = work.attention(q, k, v, mask)
torch.testing.assert_close(weights.sum(-1), torch.tensor([[[1.0, 0.0]]]))
assert output[0, 0, 1].eq(0).all()
assert torch.isfinite(output).all() and torch.isfinite(weights).all()
output.sum().backward()
assert torch.isfinite(q.grad).all() and q.grad[0, 0, 1].eq(0).all()
print("Weights:\n", weights)
print("Row sums:", weights.sum(-1))
print("Fully blocked output:", output[0, 0, 1])
print("All query gradients finite:", bool(torch.isfinite(q.grad).all()))


Weights:
 tensor([[[[0.3299, 0.3067, 0.3634],
          [0.0000, 0.0000, 0.0000]]]], grad_fn=<WhereBackward0>)
Row sums: tensor([[[1.0000, 0.0000]]], grad_fn=<SumBackward1>)
Fully blocked output: tensor([0., 0., 0., 0., 0.], grad_fn=<SelectBackward0>)
All query gradients finite: True


## 2C — Shift targets and exclude padded labels explicitly

BOS starts the decoder prefix and EOS remains a next-token target. In a padded batch, labels equal to PAD=6 are ignored by this loss. Cross-entropy is the mean over the remaining valid targets, not over all positions.

The gradient at an ignored target must be zero. This loss mask does **not** prevent attention to padding keys; the earlier padding-key example handled that separate information path.

In [7]:
ids = torch.tensor([[4, 0, 1, 2, 5], [4, 2, 0, 5, 6]])
inputs, labels = work.next_token_pairs(ids)
g = torch.Generator().manual_seed(306)
logits = torch.randn(2, 4, 7, generator=g, requires_grad=True)
loss = work.masked_cross_entropy(logits, labels, ignore_index=6)
valid = labels.ne(6)
reference = F.cross_entropy(logits[valid], labels[valid])
torch.testing.assert_close(loss, reference)
loss.backward()
assert logits.grad[~valid].eq(0).all()
print("Decoder inputs:\n", inputs)
print("Next-token labels:\n", labels)
print("Valid supervised targets:", int(valid.sum()))
print("Padding-aware cross-entropy:", float(loss.detach()))
print("Largest ignored-target gradient:", float(logits.grad[~valid].abs().max()))


Decoder inputs:
 tensor([[4, 0, 1, 2],
        [4, 2, 0, 5]])
Next-token labels:
 tensor([[0, 1, 2, 5],
        [2, 0, 5, 6]])
Valid supervised targets: 7
Padding-aware cross-entropy: 2.623682737350464
Largest ignored-target gradient: 0.0


## Empty supervision and interpreting a successful run

If every target is ignored, this API returns differentiable zero loss with zero gradients. That is a declared training-data edge policy, not evidence of a learned model or correct predictions.

A successful reference run establishes these computations and stated invariants for the checked cases. For independent learning, solve fresh tasks in **STUDENT_PRACTICE.md** and explain your own code. For a later application, overfit one tiny batch as a debugging step, then compare a simple baseline and actual generated outputs on a separate held-out split.

In [8]:
logits = torch.randn(1, 2, 7, requires_grad=True)
labels = torch.full((1, 2), 6)
loss = work.masked_cross_entropy(logits, labels, ignore_index=6)
assert loss.item() == 0.0 and loss.requires_grad
loss.backward()
assert logits.grad.eq(0).all()
print("All-ignored loss:", float(loss.detach()))
print("Gradient absolute sum:", float(logits.grad.abs().sum()))
print("Completed checks: reference attention, shapes, causality, offsets, padding, empty rows, shift and loss.")


All-ignored loss: 0.0
Gradient absolute sum: 0.0
Completed checks: reference attention, shapes, causality, offsets, padding, empty rows, shift and loss.
